In [1]:
!git clone https://github.com/VuTrinhNguyenHoang/incremental-blood-cell-classification.git
%cd incremental-blood-cell-classification
%ls -l

Cloning into 'incremental-blood-cell-classification'...
remote: Enumerating objects: 220, done.
remote: Counting objects: 100% (220/220), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 220 (delta 105), reused 171 (delta 74), pack-reused 0 (from 0)
Receiving objects: 100% (220/220), 51.85 KiB | 1.99 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/kaggle/working/incremental-blood-cell-classification
total 24
-rw-r--r-- 1 root root 1084 Aug  7 13:37 LICENSE
drwxr-xr-x 2 root root 4096 Aug  7 13:37 notebooks/
-rw-r--r-- 1 root root  502 Aug  7 13:37 pyproject.toml
-rw-r--r-- 1 root root   39 Aug  7 13:37 README.md
drwxr-xr-x 3 root root 4096 Aug  7 13:37 src/
drwxr-xr-x 3 root root 4096 Aug  7 13:37 tests/


In [2]:
%pip install -q .

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 2.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import gc
import json
import time
from pathlib import Path

import torch

from incremental_blood_cell.config import ExperimentConfig
from incremental_blood_cell.data import load_bloodmnist
from incremental_blood_cell.experiment import run_experiment

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
Device: cuda
GPU: Tesla T4


In [4]:
data_root = Path("/kaggle/working/data")

train_dataset = load_bloodmnist(
    split="train",
    root=data_root,
    download=True,
)

validation_dataset = load_bloodmnist(
    split="val",
    root=data_root,
    download=True,
)

print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))

100%|██████████| 156M/156M [00:45<00:00, 3.47MB/s]


Train: 11959
Validation: 1712


In [5]:
methods = (
    "joint",
    "finetuning",
    "lwf",
    "random_replay",
    "replay_kd",
    "hybrid",
)

class_orders = (
    (0, 1, 2, 3, 4, 5, 6, 7),
    (4, 6, 1, 7, 0, 3, 2, 5),
    (2, 5, 7, 0, 6, 1, 4, 3),
)

pilot_order = class_orders[0]
pilot_seed = 0

In [6]:
output_dir = Path("/kaggle/working/outputs/pilot")
output_dir.mkdir(parents=True, exist_ok=True)

In [7]:
results = {}

for method in methods:
    run_id = f"{method}_order0_seed{pilot_seed}"
    result_path = output_dir / f"{run_id}.json"

    if result_path.exists():
        print(f"Skipping completed run: {run_id}")
        continue

    config = ExperimentConfig(
        method=method,
        class_order=pilot_order,
        seed=pilot_seed,
        epochs=10,
        batch_size=128,
        learning_rate=1e-3,
        weight_decay=1e-4,
        memory_size=160,
        distillation_weight=1.0,
        temperature=2.0,
    )

    print(f"\nRunning: {run_id}")
    start_time = time.perf_counter()

    result = run_experiment(
        config=config,
        train_dataset=train_dataset,
        test_dataset=validation_dataset,
        device=device,
        show_progress=True,
    )

    duration_seconds = time.perf_counter() - start_time

    summary = result.to_dict()
    summary["run_id"] = run_id
    summary["evaluation_split"] = "validation"
    summary["duration_seconds"] = duration_seconds

    with result_path.open("w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)

    print(
        {
            "run_id": run_id,
            "final_average_accuracy": result.final_average_accuracy,
            "average_forgetting": result.average_forgetting,
            "backward_transfer": result.backward_transfer,
            "duration_minutes": duration_seconds / 60,
        }
    )

    result.model.to("cpu")
    del result

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Running: joint_order0_seed0
Experience 1/3 | classes=(0, 1, 2, 3) | train_samples=6144


Training:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Experience 3/3 | loss=0.0779 | avg_acc=0.8645 | memory=160 | forgetting=0.1525
{'run_id': 'hybrid_order0_seed0', 'final_average_accuracy': 0.864504265380205, 'average_forgetting': 0.1525146500096593, 'backward_transfer': -0.1525146500096593, 'duration_minutes': 8.880506430400002}


In [8]:
records = []

for result_path in sorted(output_dir.glob("*.json")):
    with result_path.open(encoding="utf-8") as file:
        records.append(json.load(file))

for record in records:
    print(
        record["config"]["method"],
        {
            "final_accuracy": round(
                record["final_average_accuracy"],
                4,
            ),
            "forgetting": round(
                record["average_forgetting"],
                4,
            ),
            "bwt": round(
                record["backward_transfer"],
                4,
            ),
            "minutes": round(
                record["duration_seconds"] / 60,
                2,
            ),
        },
    )

assert len(records) == len(methods)
print("Pilot experiments completed.")

finetuning {'final_accuracy': 0.3333, 'forgetting': 0.9539, 'bwt': -0.9539, 'minutes': 7.56}
hybrid {'final_accuracy': 0.8645, 'forgetting': 0.1525, 'bwt': -0.1525, 'minutes': 8.88}
joint {'final_accuracy': 0.8187, 'forgetting': 0.1493, 'bwt': -0.1124, 'minutes': 16.39}
lwf {'final_accuracy': 0.3327, 'forgetting': 0.9502, 'bwt': -0.9502, 'minutes': 8.44}
random_replay {'final_accuracy': 0.8584, 'forgetting': 0.1367, 'bwt': -0.1367, 'minutes': 7.78}
replay_kd {'final_accuracy': 0.8326, 'forgetting': 0.1491, 'bwt': -0.1491, 'minutes': 8.69}
Pilot experiments completed.
